# FOIA intake assistant: walkthrough

A runnable tour of the prototype: how a request becomes a case packet, how the data-handling gateway decides what a model may see, and how to point the same code at the OpenAI API, a government-cloud deployment, or an on-prem server.

Everything below runs **offline** with no API key. Cells that call a real endpoint are marked and skipped unless you set the key.

*All records and the agency are fictional.*

In [1]:
import sys, json, os
sys.path.insert(0, '..')

from foia_assist.corpus import load
from foia_assist.pipeline import Assistant
from foia_assist.gateway import AuditLog, Gateway
from foia_assist import policy, prescreen

docs = load()
print(f'{len(docs)} records')
for level in policy.LEVELS:
    print(f'  {level:<9}', sum(d.marking == level for d in docs))

16 records
  PUBLIC    5
  INTERNAL  9
  CUI       2


## 1. A request becomes a case packet

The offline endpoint uses deterministic rules for triage. It is the air-gapped mode and the baseline a model has to beat.

In [2]:
assistant = Assistant(docs, audit=AuditLog(None))
packet = assistant.process('Any audit or investigation of improper payments in the broadband program.', 'media')
print(json.dumps(packet['triage'], indent=2))
print('fee category:', packet['fee_category'])
for c in packet['candidates']:
    flags = sorted({f['exemption'] for f in c['flags']})
    print(f"{c['id']} [{c['marking']:<8}] score {c['score']:>6}  flags {flags}")

{
  "search_query": "audit investigation improper payments broadband program",
  "programs": [
    "Rural Broadband"
  ],
  "start_year": null,
  "end_year": null,
  "track": "complex",
  "needs_clarification": false,
  "clarification_question": null
}
fee category: news media
FGA-005 [CUI     ] score 17.919  flags ['b(6)', 'b(7)(A)', 'b(7)(E)', 'review']
FGA-009 [PUBLIC  ] score  3.889  flags []
FGA-001 [PUBLIC  ] score   3.85  flags []
FGA-003 [INTERNAL] score   2.31  flags []
FGA-013 [PUBLIC  ] score  1.348  flags []
FGA-014 [INTERNAL] score  1.062  flags ['b(5)']


## 2. Exemption pre-screen

Candidate flags with the exact span and a reason. A FOIA officer decides; the tool only points.

In [3]:
audit_report = next(d for d in docs if d.id == 'FGA-005')
for f in prescreen.screen(audit_report):
    print(f'{f.exemption:<8} {f.reason:<45} {f.text!r}')
print()
print(prescreen.redact(audit_report, prescreen.screen(audit_report)))

review   Document is marked CUI; requires CUI/originator review before release ''
b(7)(A)  Open law-enforcement matter                   'criminal investigation'
b(7)(A)  Open law-enforcement matter                   'OIG case 25-0417'
b(6)     Social Security number                        '219-09-9999'
b(6)     Date of birth                                 'date of birth 04/11/1971'
b(7)(E)  Investigative techniques or detection methods 'investigative techniques described in Appendix C'

CUI//PRVCY//LES. The audit identified $3.1 million in payments to one grantee that lacked supporting documentation. The matter has been referred for [b(7)(A)] ([b(7)(A)]). The subject's identifiers: SSN [b(6)], [b(6)]. Law enforcement sensitive: do not disclose [b(7)(E)].


## 3. The gateway: what a model is allowed to see

Here a fake client stands in for a model endpoint authorized for **PUBLIC** content only. We run the same request and look at what was actually sent.

In [4]:
class Recorder:
    def __init__(self): self.sent = []
    def complete_json(self, system, user, schema, name):
        self.sent.append(user)
        if name == 'foia_triage':
            from foia_assist.triage import rules
            return rules(user.split('<request>\n')[1].split('\n</request>')[0]), {}
        return {'responsive': 'possibly', 'reason': '(fake model)'}, {}

rec, audit = Recorder(), AuditLog(None)
eps = {'default': 'pub', 'endpoints': {'pub': {'name': 'pub', 'kind': 'openai', 'authorized_level': 'PUBLIC'}}}
p = Assistant(docs, eps, audit, {'pub': rec}).process('Any audit or investigation of improper payments in the broadband program.', 'media')
for c in p['candidates']:
    print(f"{c['id']} [{c['marking']:<8}] -> {c['note_source']}")
print('\nSSN in anything sent to the model?', any('219-09-9999' in s for s in rec.sent))

FGA-005 [CUI     ] -> withheld
FGA-009 [PUBLIC  ] -> model
FGA-001 [PUBLIC  ] -> model
FGA-003 [INTERNAL] -> withheld
FGA-013 [PUBLIC  ] -> model
FGA-014 [INTERNAL] -> withheld

SSN in anything sent to the model? False


In [5]:
for r in audit.records:
    print(r['event'], {k: r[k] for k in ('endpoint', 'content_level', 'sources', 'source') if k in r})

request_received {'endpoint': 'pub'}
model_call {'endpoint': 'pub', 'content_level': 'PUBLIC', 'sources': ['request_text']}
record_withheld_from_model {'endpoint': 'pub', 'content_level': 'CUI', 'source': 'FGA-005'}
model_call {'endpoint': 'pub', 'content_level': 'PUBLIC', 'sources': ['request_text', 'FGA-009']}
model_call {'endpoint': 'pub', 'content_level': 'PUBLIC', 'sources': ['request_text', 'FGA-001']}
record_withheld_from_model {'endpoint': 'pub', 'content_level': 'INTERNAL', 'source': 'FGA-003'}
model_call {'endpoint': 'pub', 'content_level': 'PUBLIC', 'sources': ['request_text', 'FGA-013']}
record_withheld_from_model {'endpoint': 'pub', 'content_level': 'INTERNAL', 'source': 'FGA-014'}
packet_created {}


Trying to push CUI at that endpoint directly is refused and logged:

In [6]:
gw = Gateway(eps['endpoints']['pub'], audit, rec)
try:
    gw.ask('DEMO', 'attempt', 'system', audit_report.body, {}, sources=[(audit_report.id, audit_report.marking)])
except policy.PolicyViolation as e:
    print('blocked:', e)
print(audit.records[-1]['event'])

blocked: CUI content cannot be sent to endpoint 'pub' (authorized up to PUBLIC)
model_call_blocked


## 4. Evals

The same suite runs in CI and blocks a deploy if a gate fails.

In [7]:
import subprocess
print(subprocess.run([sys.executable, '-m', 'evals.run_evals'], cwd='..', capture_output=True, text=True).stdout)


FOIA assistant eval · 12 requests · 16 records

Release gates
  [PASS] recall_at_5                 100.0%   (target >= 90%)
  [PASS] track_accuracy              100.0%   (target >= 80%)
  [PASS] clarification_accuracy      100.0%   (target >= 100%)
  [PASS] high_severity_pii_recall    100.0%   (target >= 100%)
  [PASS] pii_recall                   87.5%   (target >= 85%)
  [PASS] leaked_records                   0   (target <= 0)
  [PASS] unblocked_violations             0   (target <= 0)

  program_accuracy           100.0%
  pii_precision              87.5%
  model calls in leak test   39  ·  records withheld 31  ·  blocked attempts logged 11

PII pre-screen notes
  extra flag in FGA-004: '(202) 555-0143'
  missed in FGA-008: 'Laura Chen'

ALL GATES PASSED




## 5. Pointing at a real endpoint

Endpoints live in `config/endpoints.json`. The code path is identical; only the URL, auth header and `authorized_level` change. This cell runs only if `OPENAI_API_KEY` is set.

In [8]:
if os.environ.get('OPENAI_API_KEY'):
    live = Assistant(docs)
    p = live.process('The 2025 notice of funding opportunity for the water infrastructure program.', 'educational', endpoint='commercial-cloud')
    for c in p['candidates']:
        print(c['id'], c['marking'], c['note_source'], c['note'])
else:
    print('OPENAI_API_KEY not set; skipped. Records above PUBLIC would show note_source=withheld on this endpoint.')

OPENAI_API_KEY not set; skipped. Records above PUBLIC would show note_source=withheld on this endpoint.
